In [1]:
"""
PDE:
    u_t + a u_x = nu u_xx,   (x,t) in [0,1] x [0,0.5]

Parameter ranges:
    a  in [0.5, 1.0]
    nu in [0.01, 0.05]

Exact benchmark family used for IC/BC/evaluation:
    u(x,t;a,nu) = 1/sqrt(4t+1) * exp( -(x - 0.2 - a t)^2 / (nu (4t+1)) )

"""

import os
import csv
import math
import random
import argparse
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


# ============================================================
# Reproducibility
# ============================================================
SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# ============================================================
# Config
# ============================================================
@dataclass
class ProblemConfig:
    x_min: float = 0.0
    x_max: float = 1.0
    t_min: float = 0.0
    t_max: float = 0.5
    a_min: float = 0.5
    a_max: float = 1.0
    nu_min: float = 0.01
    nu_max: float = 0.05
    nu_easy: float = 0.03
    x0_center: float = 0.2


CFG = ProblemConfig()


# ============================================================
# Exact solution / IC
# ============================================================
def exact_advecdiff_torch(x: torch.Tensor, t: torch.Tensor, a: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    four_t1 = 4.0 * t + 1.0
    return torch.rsqrt(four_t1) * torch.exp(-((x - CFG.x0_center - a * t) ** 2) / (nu * four_t1))


def exact_advecdiff_np(x: np.ndarray, t: np.ndarray, a: float, nu: float) -> np.ndarray:
    four_t1 = 4.0 * t + 1.0
    return (1.0 / np.sqrt(four_t1)) * np.exp(-((x - CFG.x0_center - a * t) ** 2) / (nu * four_t1))


def initial_condition_torch(x: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
    # exact solution at t=0
    return torch.exp(-((x - CFG.x0_center) ** 2) / nu)


# ============================================================
# Fourier features in time
# ============================================================
def time_fourier_features(t: torch.Tensor, num_freqs: int = 4, t_scale: float = 1.0) -> torch.Tensor:
    """
    t: (...,1)
    returns [..., 1 + 2*num_freqs]
    """
    feats = [t]
    for k in range(1, num_freqs + 1):
        omega = 2.0 * math.pi * k / t_scale
        feats.append(torch.sin(omega * t))
        feats.append(torch.cos(omega * t))
    return torch.cat(feats, dim=-1)


# ============================================================
# Model
# ============================================================
class DynamicBasisMetaSPINN_AdvecDiff(nn.Module):
    def __init__(
        self,
        M: int = 48,
        hidden: int = 128,
        num_freqs: int = 4,
        h_min: float = 0.015,
        h_max: float = 0.35,
    ):
        super().__init__()
        self.M = M
        self.num_freqs = num_freqs
        self.h_min = h_min
        self.h_max = h_max

        in_dim = 1 + 2 * num_freqs + 2  # time Fourier + (a, nu)

        self.temporal_net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 3 * M),
        )

        # Learned static spatial anchor distribution.
        # The time/parameter network predicts offsets around these anchors.
        self.anchor_raw = nn.Parameter(torch.linspace(-2.0, 2.0, M))
        self.amp_scale = nn.Parameter(0.1 * torch.ones(M))

    def normalize_params(self, a: torch.Tensor, nu: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        a_mean = 0.5 * (CFG.a_min + CFG.a_max)
        a_std = max(0.5 * (CFG.a_max - CFG.a_min), 1e-8)
        nu_mean = 0.5 * (CFG.nu_min + CFG.nu_max)
        nu_std = max(0.5 * (CFG.nu_max - CFG.nu_min), 1e-8)
        a_n = (a - a_mean) / a_std
        nu_n = (nu - nu_mean) / nu_std
        return a_n, nu_n

    def dynamic_params(self, t: torch.Tensor, a: torch.Tensor, nu: torch.Tensor):
        """
        t, a, nu: (N,1)
        returns alpha, xi, h each of shape (N,M)
        """
        a_n, nu_n = self.normalize_params(a, nu)
        t_feats = time_fourier_features(t, num_freqs=self.num_freqs, t_scale=max(CFG.t_max, 1e-8))
        inp = torch.cat([t_feats, a_n, nu_n], dim=-1)

        out = self.temporal_net(inp).view(-1, 3, self.M)
        amp_raw = out[:, 0, :]
        xi_raw = out[:, 1, :]
        h_raw = out[:, 2, :]

        # Static anchors in [x_min, x_max]
        anchors = CFG.x_min + (CFG.x_max - CFG.x_min) * torch.sigmoid(self.anchor_raw).view(1, self.M)

        # Dynamic amplitudes
        alpha = torch.tanh(amp_raw) * self.amp_scale.view(1, self.M)

        # Dynamic centers in physical domain
        xi = anchors + 0.35 * torch.tanh(xi_raw)
        xi = torch.clamp(xi, CFG.x_min, CFG.x_max)

        # Dynamic widths in a safe bounded range
        h = self.h_min + (self.h_max - self.h_min) * torch.sigmoid(h_raw)

        return alpha, xi, h

    def basis_eval(self, x: torch.Tensor, t: torch.Tensor, a: torch.Tensor, nu: torch.Tensor) -> torch.Tensor:
        alpha, xi, h = self.dynamic_params(t, a, nu)
        x_exp = x.expand(-1, self.M)
        phi = torch.exp(-0.5 * ((x_exp - xi) / h) ** 2)
        v = torch.sum(alpha * phi, dim=1, keepdim=True)
        return v, (alpha, xi, h)

    def forward(self, xtanu: torch.Tensor, return_aux: bool = False):
        x = xtanu[:, 0:1]
        t = xtanu[:, 1:2]
        a = xtanu[:, 2:3]
        nu = xtanu[:, 3:4]

        u0 = initial_condition_torch(x, nu)
        v, aux = self.basis_eval(x, t, a, nu)
        u = u0 + t * v

        if return_aux:
            return u.squeeze(-1), aux
        return u.squeeze(-1)


# ============================================================
# Sampling utilities
# ============================================================
def sample_a_nu(N: int, nu_range: Tuple[float, float], device: torch.device):
    a = CFG.a_min + (CFG.a_max - CFG.a_min) * torch.rand(N, 1, device=device)
    # log-uniform for nu is better for resolving the lower-viscosity cases
    u = torch.rand(N, 1, device=device)
    log_min = math.log10(nu_range[0])
    log_max = math.log10(nu_range[1])
    nu = 10 ** (log_min + (log_max - log_min) * u)
    return a, nu


def sample_interior_points(N: int, nu_range: Tuple[float, float], device: torch.device):
    x = CFG.x_min + (CFG.x_max - CFG.x_min) * torch.rand(N, 1, device=device)
    t = CFG.t_min + (CFG.t_max - CFG.t_min) * torch.rand(N, 1, device=device)
    a, nu = sample_a_nu(N, nu_range, device)
    return torch.cat([x, t, a, nu], dim=1)


def sample_localized_interior_points(N: int, nu_range: Tuple[float, float], device: torch.device):
    """
    Local points near the evolving Gaussian packet center.
    No characteristics are used in the model, but for sampling we may use the known
    benchmark center to focus collocation where the solution is active.
    This is similar in spirit to adaptive collocation around localized features.
    """
    t = CFG.t_min + (CFG.t_max - CFG.t_min) * torch.rand(N, 1, device=device)
    a, nu = sample_a_nu(N, nu_range, device)

    center = CFG.x0_center + a * t
    width = 0.6 * torch.sqrt(nu * (4.0 * t + 1.0) + 1e-12)
    x = center + width * torch.randn_like(center)
    x = torch.clamp(x, CFG.x_min, CFG.x_max)
    return torch.cat([x, t, a, nu], dim=1)


def sample_ic_points(N: int, nu_range: Tuple[float, float], device: torch.device):
    x = CFG.x_min + (CFG.x_max - CFG.x_min) * torch.rand(N, 1, device=device)
    t = torch.zeros(N, 1, device=device)
    a, nu = sample_a_nu(N, nu_range, device)
    return torch.cat([x, t, a, nu], dim=1)


def sample_bc_points(N: int, nu_range: Tuple[float, float], device: torch.device):
    t = CFG.t_min + (CFG.t_max - CFG.t_min) * torch.rand(N, 1, device=device)
    a, nu = sample_a_nu(N, nu_range, device)
    xL = torch.full((N, 1), CFG.x_min, device=device)
    xR = torch.full((N, 1), CFG.x_max, device=device)
    left = torch.cat([xL, t, a, nu], dim=1)
    right = torch.cat([xR, t, a, nu], dim=1)
    return left, right


# ============================================================
# PDE residual
# ============================================================
def advecdiff_residual(model: nn.Module, xtanu: torch.Tensor) -> torch.Tensor:
    xtanu = xtanu.clone().detach().requires_grad_(True)
    u = model(xtanu)

    grads = torch.autograd.grad(
        outputs=u,
        inputs=xtanu,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
        retain_graph=True,
    )[0]

    u_x = grads[:, 0]
    u_t = grads[:, 1]

    grads_x = torch.autograd.grad(
        outputs=u_x,
        inputs=xtanu,
        grad_outputs=torch.ones_like(u_x),
        create_graph=True,
        retain_graph=True,
    )[0]
    u_xx = grads_x[:, 0]

    a = xtanu[:, 2]
    nu = xtanu[:, 3]

    return u_t + a * u_x - nu * u_xx




# ============================================================
# Predictor-guided corrector (frozen spacetime RBF + ridge LS)
# ============================================================
def extract_dynamic_centers_from_predictor(
    model,
    a_test: float,
    nu_test: float,
    device,
    n_time_guides: int = 16,
    n_dyn_keep: int = 10,
    sigma_t_scale: float = 1.25,
):
    """
    Use the trained predictor only as a geometry generator.
    At each guide time, keep the strongest dynamic centers according to |alpha|.
    """
    t_vals = torch.linspace(CFG.t_min, CFG.t_max, n_time_guides, device=device).view(-1, 1)
    a_vals = torch.full_like(t_vals, float(a_test))
    nu_vals = torch.full_like(t_vals, float(nu_test))

    model.eval()
    with torch.no_grad():
        alpha, xi, h = model.dynamic_params(t_vals, a_vals, nu_vals)

    alpha = alpha.detach().cpu().numpy()
    xi = xi.detach().cpu().numpy()
    h = h.detach().cpu().numpy()
    t_np = t_vals[:, 0].detach().cpu().numpy()
    dt = (CFG.t_max - CFG.t_min) / max(n_time_guides - 1, 1)

    mu_x, mu_t, sigma_x, sigma_t = [], [], [], []
    for k, tval in enumerate(t_np):
        idx = np.argsort(np.abs(alpha[k]))[::-1][:n_dyn_keep]
        for j in idx:
            mu_x.append(float(xi[k, j]))
            mu_t.append(float(tval))
            sigma_x.append(float(h[k, j]))
            sigma_t.append(float(max(sigma_t_scale * dt, 1e-3)))

    return (
        np.asarray(mu_x),
        np.asarray(mu_t),
        np.asarray(sigma_x),
        np.asarray(sigma_t),
    )


def extract_gradient_refinement_centers(
    model,
    a_test: float,
    nu_test: float,
    device,
    n_time_guides: int = 16,
    n_grad_keep: int = 10,
    grad_grid_x: int = 320,
    sigma_x_scale: float = 0.6,
    sigma_t_scale: float = 0.8,
):
    """
    Place additional narrow centers where the predictor has large |u_x|.
    """
    x_grid = np.linspace(CFG.x_min, CFG.x_max, grad_grid_x)
    t_guides = np.linspace(CFG.t_min, CFG.t_max, n_time_guides)
    dx = (CFG.x_max - CFG.x_min) / max(grad_grid_x - 1, 1)
    dt = (CFG.t_max - CFG.t_min) / max(n_time_guides - 1, 1)

    mu_x, mu_t, sigma_x, sigma_t = [], [], [], []
    model.eval()

    for tval in t_guides:
        xtanu = np.stack(
            [
                x_grid,
                tval * np.ones_like(x_grid),
                a_test * np.ones_like(x_grid),
                nu_test * np.ones_like(x_grid),
            ],
            axis=1,
        )
        xtanu_t = torch.tensor(xtanu, dtype=torch.float32, device=device, requires_grad=True)
        u_pred = model(xtanu_t)
        grads = torch.autograd.grad(
            outputs=u_pred,
            inputs=xtanu_t,
            grad_outputs=torch.ones_like(u_pred),
            create_graph=False,
            retain_graph=False,
        )[0]
        ux = grads[:, 0].detach().cpu().numpy()
        score = np.abs(ux)

        idx_all = np.argsort(score)[::-1]
        picked = []
        for j in idx_all:
            xj = float(x_grid[j])
            if all(abs(xj - xp) > 4.0 * dx for xp in picked):
                picked.append(xj)
            if len(picked) >= n_grad_keep:
                break

        for xj in picked:
            mu_x.append(xj)
            mu_t.append(float(tval))
            sigma_x.append(float(max(1.5 * dx, sigma_x_scale * nu_test)))
            sigma_t.append(float(max(0.35 * dt, sigma_t_scale * dt, 1e-3)))

    return (
        np.asarray(mu_x),
        np.asarray(mu_t),
        np.asarray(sigma_x),
        np.asarray(sigma_t),
    )


def build_fixed_background_geometry(
    n_fix_x: int = 18,
    n_fix_t: int = 12,
    sigma_x_factor: float = 1.15,
    sigma_t_factor: float = 1.15,
):
    """
    Uniform spacetime background dictionary.
    """
    x_fix = np.linspace(CFG.x_min, CFG.x_max, n_fix_x)
    t_fix = np.linspace(CFG.t_min, CFG.t_max, n_fix_t)
    XX, TT = np.meshgrid(x_fix, t_fix, indexing="ij")

    dx = (CFG.x_max - CFG.x_min) / max(n_fix_x - 1, 1)
    dt = (CFG.t_max - CFG.t_min) / max(n_fix_t - 1, 1)

    mu_x = XX.ravel()
    mu_t = TT.ravel()
    sigma_x = (sigma_x_factor * dx) * np.ones_like(mu_x)
    sigma_t = (sigma_t_factor * dt) * np.ones_like(mu_t)
    return mu_x, mu_t, sigma_x, sigma_t


def build_refined_frozen_hidden_geometry(
    model,
    a_test: float,
    nu_test: float,
    device,
    n_time_guides: int = 16,
    n_dyn_keep: int = 10,
    n_grad_keep: int = 10,
    n_fix_x: int = 18,
    n_fix_t: int = 12,
):
    dyn = extract_dynamic_centers_from_predictor(
        model=model,
        a_test=a_test,
        nu_test=nu_test,
        device=device,
        n_time_guides=n_time_guides,
        n_dyn_keep=n_dyn_keep,
        sigma_t_scale=1.25,
    )
    grad = extract_gradient_refinement_centers(
        model=model,
        a_test=a_test,
        nu_test=nu_test,
        device=device,
        n_time_guides=n_time_guides,
        n_grad_keep=n_grad_keep,
        grad_grid_x=320,
        sigma_x_scale=0.6,
        sigma_t_scale=0.8,
    )
    fix = build_fixed_background_geometry(
        n_fix_x=n_fix_x,
        n_fix_t=n_fix_t,
        sigma_x_factor=1.15,
        sigma_t_factor=1.15,
    )

    mu_x = np.concatenate([dyn[0], grad[0], fix[0]], axis=0)
    mu_t = np.concatenate([dyn[1], grad[1], fix[1]], axis=0)
    sigma_x = np.concatenate([dyn[2], grad[2], fix[2]], axis=0)
    sigma_t = np.concatenate([dyn[3], grad[3], fix[3]], axis=0)

    return {
        "mu_x": mu_x,
        "mu_t": mu_t,
        "sigma_x": np.maximum(sigma_x, 1e-4),
        "sigma_t": np.maximum(sigma_t, 1e-4),
    }


def eval_spacetime_rbf_np(x, t, mu_x, mu_t, sigma_x, sigma_t):
    x = np.asarray(x).reshape(-1, 1)
    t = np.asarray(t).reshape(-1, 1)
    dx = x - np.asarray(mu_x).reshape(1, -1)
    dt = t - np.asarray(mu_t).reshape(1, -1)
    sx = np.asarray(sigma_x).reshape(1, -1)
    st = np.asarray(sigma_t).reshape(1, -1)
    return np.exp(-(dx ** 2) / (2.0 * sx ** 2) - (dt ** 2) / (2.0 * st ** 2))


def eval_spacetime_rbf_derivatives_np(x, t, mu_x, mu_t, sigma_x, sigma_t):
    x = np.asarray(x).reshape(-1, 1)
    t = np.asarray(t).reshape(-1, 1)
    dx = x - np.asarray(mu_x).reshape(1, -1)
    dt = t - np.asarray(mu_t).reshape(1, -1)
    sx = np.asarray(sigma_x).reshape(1, -1)
    st = np.asarray(sigma_t).reshape(1, -1)

    Phi = np.exp(-(dx ** 2) / (2.0 * sx ** 2) - (dt ** 2) / (2.0 * st ** 2))
    Phi_x = -(dx / (sx ** 2)) * Phi
    Phi_xx = ((dx ** 2) / (sx ** 4) - 1.0 / (sx ** 2)) * Phi
    Phi_t = -(dt / (st ** 2)) * Phi
    return Phi, Phi_x, Phi_xx, Phi_t


def solve_corrector_for_case(
    a_test: float,
    nu_test: float,
    hidden_geom,
    N_res: int = 6000,
    N_bc: int = 1200,
    lambda_ridge: float = 1e-8,
    seed: int = 1234,
):
    """
    Solve for the outer coefficients c_j in
        u(x,t) = u0(x) + t * sum_j c_j phi_j(x,t)

    PDE: u_t + a u_x - nu u_xx = 0
    """
    rng = np.random.default_rng(seed)
    mu_x = hidden_geom["mu_x"]
    mu_t = hidden_geom["mu_t"]
    sigma_x = hidden_geom["sigma_x"]
    sigma_t = hidden_geom["sigma_t"]
    M = len(mu_x)

    x_res = CFG.x_min + (CFG.x_max - CFG.x_min) * rng.random(N_res)
    t_res = CFG.t_min + (CFG.t_max - CFG.t_min) * rng.random(N_res)

    Phi, Phi_x, Phi_xx, Phi_t = eval_spacetime_rbf_derivatives_np(
        x_res, t_res, mu_x, mu_t, sigma_x, sigma_t
    )

    u0_x = -(2.0 * (x_res - CFG.x0_center) / nu_test) * np.exp(-((x_res - CFG.x0_center) ** 2) / nu_test)
    u0_xx = ((4.0 * (x_res - CFG.x0_center) ** 2) / (nu_test ** 2) - (2.0 / nu_test)) * np.exp(
        -((x_res - CFG.x0_center) ** 2) / nu_test
    )

    A_res = Phi + t_res[:, None] * Phi_t + (a_test * t_res[:, None]) * Phi_x - (nu_test * t_res[:, None]) * Phi_xx
    b_res = -(a_test * u0_x - nu_test * u0_xx)

    t_bc = CFG.t_min + (CFG.t_max - CFG.t_min) * rng.random(N_bc)

    xL = CFG.x_min * np.ones(N_bc)
    xR = CFG.x_max * np.ones(N_bc)

    PhiL = eval_spacetime_rbf_np(xL, t_bc, mu_x, mu_t, sigma_x, sigma_t)
    PhiR = eval_spacetime_rbf_np(xR, t_bc, mu_x, mu_t, sigma_x, sigma_t)

    uL_true = exact_advecdiff_np(xL, t_bc, a_test, nu_test)
    uR_true = exact_advecdiff_np(xR, t_bc, a_test, nu_test)
    u0L = np.exp(-((xL - CFG.x0_center) ** 2) / nu_test)
    u0R = np.exp(-((xR - CFG.x0_center) ** 2) / nu_test)

    A_bc_L = t_bc[:, None] * PhiL
    b_bc_L = uL_true - u0L
    A_bc_R = t_bc[:, None] * PhiR
    b_bc_R = uR_true - u0R

    A = np.vstack([A_res, A_bc_L, A_bc_R])
    b = np.concatenate([b_res, b_bc_L, b_bc_R])

    A_aug = np.vstack([A, math.sqrt(lambda_ridge) * np.eye(M)])
    b_aug = np.concatenate([b, np.zeros(M)])
    coeffs = np.linalg.lstsq(A_aug, b_aug, rcond=None)[0]

    return coeffs, {"A_shape": A.shape, "n_hidden": M, "lambda_ridge": lambda_ridge}


def evaluate_corrected_case_on_grid(
    a_test: float,
    nu_test: float,
    hidden_geom,
    coeffs,
    Nx: int = 200,
    Nt: int = 200,
):
    x = np.linspace(CFG.x_min, CFG.x_max, Nx)
    t = np.linspace(CFG.t_min, CFG.t_max, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")

    Phi = eval_spacetime_rbf_np(
        X.ravel(),
        T.ravel(),
        hidden_geom["mu_x"],
        hidden_geom["mu_t"],
        hidden_geom["sigma_x"],
        hidden_geom["sigma_t"],
    )
    correction = (T.ravel()[:, None] * Phi) @ coeffs
    u0 = np.exp(-((X.ravel() - CFG.x0_center) ** 2) / nu_test)
    u_corr = (u0 + correction).reshape(Nx, Nt)

    u_exact = exact_advecdiff_np(X, T, a_test, nu_test)
    abs_err = np.abs(u_corr - u_exact)
    rel_l2 = np.linalg.norm((u_corr - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)

    return {
        "x": x,
        "t": t,
        "X": X,
        "T": T,
        "u_exact": u_exact,
        "u_corr": u_corr,
        "abs_err": abs_err,
        "rel_l2": rel_l2,
    }


def run_corrector_on_test_cases(
    model,
    test_cases,
    Nx: int,
    Nt: int,
    save_dir: str,
    device,
    n_time_guides: int = 16,
    n_dyn_keep: int = 10,
    n_grad_keep: int = 10,
    n_fix_x: int = 18,
    n_fix_t: int = 12,
    N_res: int = 6000,
    N_bc: int = 1200,
    lambda_ridge: float = 1e-8,
):
    summary = []
    num_tests = len(test_cases)
    fig, axs = plt.subplots(4, num_tests, figsize=(4.5 * num_tests, 12.0), constrained_layout=True)
    if num_tests == 1:
        axs = axs.reshape(4, 1)

    predictor_results = []
    corrector_results = []
    hidden_counts = []

    for name, a_test, nu_test in test_cases:
        pred_res = evaluate_case_on_grid(model, a_test, nu_test, Nx=Nx, Nt=Nt, device=device)
        hidden_geom = build_refined_frozen_hidden_geometry(
            model=model,
            a_test=a_test,
            nu_test=nu_test,
            device=device,
            n_time_guides=n_time_guides,
            n_dyn_keep=n_dyn_keep,
            n_grad_keep=n_grad_keep,
            n_fix_x=n_fix_x,
            n_fix_t=n_fix_t,
        )
        coeffs, info = solve_corrector_for_case(
            a_test=a_test,
            nu_test=nu_test,
            hidden_geom=hidden_geom,
            N_res=N_res,
            N_bc=N_bc,
            lambda_ridge=lambda_ridge,
            seed=SEED,
        )
        corr_res = evaluate_corrected_case_on_grid(
            a_test=a_test,
            nu_test=nu_test,
            hidden_geom=hidden_geom,
            coeffs=coeffs,
            Nx=Nx,
            Nt=Nt,
        )

        predictor_results.append(pred_res)
        corrector_results.append(corr_res)
        hidden_counts.append(info["n_hidden"])
        summary.append(
            {
                "name": name,
                "a": a_test,
                "nu": nu_test,
                "predictor_rel_l2": pred_res["rel_l2"],
                "corrector_rel_l2": corr_res["rel_l2"],
                "n_hidden": info["n_hidden"],
            }
        )

    sol_vals = np.concatenate(
        [r["u_exact"].ravel() for r in predictor_results]
        + [r["u_pred"].ravel() for r in predictor_results]
        + [r["u_corr"].ravel() for r in corrector_results]
    )
    sol_vmin, sol_vmax = np.min(sol_vals), np.max(sol_vals)

    err_vals = np.concatenate(
        [r["abs_err"].ravel() for r in predictor_results]
        + [r["abs_err"].ravel() for r in corrector_results]
    )
    err_vmin, err_vmax = 0.0, np.max(err_vals)

    im_sol = None
    im_err = None
    for j, ((name, a_test, nu_test), pred_res, corr_res, nh) in enumerate(zip(test_cases, predictor_results, corrector_results, hidden_counts)):
        im_sol = axs[0, j].imshow(pred_res["u_exact"].T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max],
                                  aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        axs[0, j].set_title(rf"$(a,\nu)=({a_test:.2f},{nu_test:.3f})$", fontsize=14)
        axs[0, j].set_xlabel("x")
        if j == 0:
            axs[0, j].set_ylabel("Exact", fontsize=13)

        axs[1, j].imshow(pred_res["u_pred"].T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max],
                         aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        axs[1, j].set_xlabel("x")
        if j == 0:
            axs[1, j].set_ylabel("Predictor", fontsize=13)

        axs[2, j].imshow(corr_res["u_corr"].T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max],
                         aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        axs[2, j].set_xlabel("x")
        if j == 0:
            axs[2, j].set_ylabel("Corrector", fontsize=13)

        im_err = axs[3, j].imshow(corr_res["abs_err"].T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max],
                                  aspect="auto", vmin=err_vmin, vmax=err_vmax)
        axs[3, j].set_xlabel("x")
        if j == 0:
            axs[3, j].set_ylabel(r"$|u_{corr}-u_{exact}|$", fontsize=13)

        axs[2, j].text(0.02, 0.98, f"relL2={corr_res['rel_l2']:.2e}\nM={nh}",
                       transform=axs[2, j].transAxes, ha="left", va="top",
                       fontsize=10, bbox=dict(boxstyle="round", fc="white", alpha=0.8))

    cbar1 = fig.colorbar(im_sol, ax=axs[0:3, :], shrink=0.95, location="right")
    cbar1.set_label("Solution value", fontsize=12)
    cbar2 = fig.colorbar(im_err, ax=axs[3, :], shrink=0.95, location="right")
    cbar2.set_label("Absolute error", fontsize=12)

    fig.suptitle("Predictor-guided spacetime-RBF corrector", fontsize=16)
    fig_path = os.path.join(save_dir, "predictor_corrector_multicase.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    csv_path = os.path.join(save_dir, "predictor_corrector_summary.csv")
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "a", "nu", "predictor_rel_l2", "corrector_rel_l2", "n_hidden"])
        writer.writeheader()
        for row in summary:
            writer.writerow(row)

    return summary

# ============================================================
# Evaluation / plotting
# ============================================================
def plot_training_history(history: Dict[str, List[float]], save_path: str):
    plt.figure(figsize=(7, 4.5))
    plt.plot(history["epoch"], history["loss"], lw=2, label="Total")
    plt.plot(history["epoch"], history["loss_pde"], lw=1.5, label="PDE")
    plt.plot(history["epoch"], history["loss_ic"], lw=1.5, label="IC")
    plt.plot(history["epoch"], history["loss_bc"], lw=1.5, label="BC")
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training loss history")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


def evaluate_case_on_grid(model, a_test: float, nu_test: float, Nx: int = 200, Nt: int = 200, device=None):
    x = np.linspace(CFG.x_min, CFG.x_max, Nx)
    t = np.linspace(CFG.t_min, CFG.t_max, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")
    A = a_test * np.ones_like(X)
    NU = nu_test * np.ones_like(X)

    xtanu = np.stack([X.ravel(), T.ravel(), A.ravel(), NU.ravel()], axis=1)
    xtanu_t = torch.tensor(xtanu, dtype=torch.float32, device=device)

    model.eval()
    with torch.no_grad():
        u_pred_flat = model(xtanu_t)

    u_pred = u_pred_flat.cpu().numpy().reshape(Nx, Nt)
    u_exact = exact_advecdiff_np(X, T, a_test, nu_test)
    abs_err = np.abs(u_pred - u_exact)
    rel_l2 = np.linalg.norm((u_pred - u_exact).ravel()) / (np.linalg.norm(u_exact.ravel()) + 1e-14)

    return {
        "x": x,
        "t": t,
        "X": X,
        "T": T,
        "u_exact": u_exact,
        "u_pred": u_pred,
        "abs_err": abs_err,
        "rel_l2": rel_l2,
    }


def plot_multicase_fields(model, test_cases, Nx: int, Nt: int, save_dir: str, device):
    num_tests = len(test_cases)
    fig, axs = plt.subplots(3, num_tests, figsize=(4.4 * num_tests, 9.6), constrained_layout=True)
    if num_tests == 1:
        axs = axs.reshape(3, 1)

    results = []
    for name, a_test, nu_test in test_cases:
        results.append((name, a_test, nu_test, evaluate_case_on_grid(model, a_test, nu_test, Nx=Nx, Nt=Nt, device=device)))

    all_sol_vals = np.concatenate([r[3]["u_exact"].ravel() for r in results] + [r[3]["u_pred"].ravel() for r in results])
    sol_vmin, sol_vmax = np.min(all_sol_vals), np.max(all_sol_vals)
    all_err_vals = np.concatenate([r[3]["abs_err"].ravel() for r in results])
    err_vmin, err_vmax = 0.0, np.max(all_err_vals)

    summary = []
    im_sol = None
    im_err = None
    for j, (name, a_test, nu_test, res) in enumerate(results):
        u_exact = res["u_exact"]
        u_pred = res["u_pred"]
        abs_err = res["abs_err"]
        rel_l2 = res["rel_l2"]
        summary.append({"name": name, "a": a_test, "nu": nu_test, "rel_l2": rel_l2})

        ax = axs[0, j]
        im_sol = ax.imshow(u_exact.T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        ax.set_title(rf"$(a,\nu)=({a_test:.2f},{nu_test:.3f})$", fontsize=14)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Exact", fontsize=13)

        ax = axs[1, j]
        ax.imshow(u_pred.T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max], aspect="auto", vmin=sol_vmin, vmax=sol_vmax)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Prediction", fontsize=13)

        ax = axs[2, j]
        im_err = ax.imshow(abs_err.T, origin="lower", extent=[CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max], aspect="auto", vmin=err_vmin, vmax=err_vmax)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$|u_{pred}-u_{exact}|$", fontsize=13)

    cbar1 = fig.colorbar(im_sol, ax=axs[0:2, :], shrink=0.95, location="right")
    cbar1.set_label("Solution value", fontsize=12)
    cbar2 = fig.colorbar(im_err, ax=axs[2, :], shrink=0.95, location="right")
    cbar2.set_label("Absolute error", fontsize=12)

    training_text = rf"Training ranges: $a \in [{CFG.a_min:.2f},{CFG.a_max:.2f}]$, $\nu \in [{CFG.nu_min:.3f},{CFG.nu_max:.3f}]$"
    fig.text(0.5, -0.02, training_text, ha="center", va="top", fontsize=13)

    combined_path = os.path.join(save_dir, "multicase_fields.png")
    fig.savefig(combined_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return summary


def plot_time_slices(model, test_cases, t_slices, Nx: int, save_dir: str, device):
    x = np.linspace(CFG.x_min, CFG.x_max, Nx)
    for name, a_test, nu_test in test_cases:
        fig, axs = plt.subplots(1, len(t_slices), figsize=(4 * len(t_slices), 3.8), constrained_layout=True)
        if len(t_slices) == 1:
            axs = [axs]
        for k, t_val in enumerate(t_slices):
            xtanu = np.stack([x, t_val * np.ones_like(x), a_test * np.ones_like(x), nu_test * np.ones_like(x)], axis=1)
            xtanu_t = torch.tensor(xtanu, dtype=torch.float32, device=device)
            with torch.no_grad():
                u_pred = model(xtanu_t).cpu().numpy()
            u_exact = exact_advecdiff_np(x, t_val, a_test, nu_test)
            ax = axs[k]
            ax.plot(x, u_exact, lw=2, label="Exact")
            ax.plot(x, u_pred, "--", lw=2, label="Prediction")
            ax.set_title(f"t = {t_val:.2f}")
            ax.set_xlabel("x")
            ax.set_ylabel("u")
            ax.grid(True, alpha=0.3)
            if k == 0:
                ax.legend()
        fig.suptitle(f"Time slices: {name} | a={a_test:.3f}, nu={nu_test:.3f}", fontsize=13)
        save_path = os.path.join(save_dir, f"time_slices_{name}.png")
        fig.savefig(save_path, dpi=300)
        plt.close(fig)


def plot_error_summary(summary, save_dir: str):
    names = [row["name"] for row in summary]
    rel_l2 = [row["rel_l2"] for row in summary]
    plt.figure(figsize=(8, 4))
    plt.bar(names, rel_l2)
    plt.yscale("log")
    plt.ylabel("Relative L2 error")
    plt.title("Generalization across test cases")
    plt.xticks(rotation=25, ha="right")
    plt.grid(True, alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "error_summary.png"), dpi=300)
    plt.close()


def save_summary_csv(summary, save_dir: str):
    path = os.path.join(save_dir, "test_case_summary.csv")
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["name", "a", "nu", "rel_l2"])
        writer.writeheader()
        for row in summary:
            writer.writerow(row)


def visualize_dynamic_centers(model, test_cases, K: int, save_dir: str, device):
    model.eval()
    t_vals = np.linspace(CFG.t_min, CFG.t_max, K).astype(np.float32)
    for name, a_test, nu_test in test_cases:
        T = torch.tensor(t_vals[:, None], dtype=torch.float32, device=device)
        A = torch.full_like(T, a_test)
        NU = torch.full_like(T, nu_test)
        with torch.no_grad():
            alpha, xi, h = model.dynamic_params(T, A, NU)
        xi_np = xi.cpu().numpy()
        alpha_np = np.abs(alpha.cpu().numpy()) + 1e-12
        h_np = h.cpu().numpy()
        X_pts, T_pts, W_pts, S_pts = [], [], [], []
        for j in range(model.M):
            for k in range(K):
                X_pts.append(xi_np[k, j])
                T_pts.append(t_vals[k])
                W_pts.append(alpha_np[k, j])
                S_pts.append(250.0 * h_np[k, j])
        fig, ax = plt.subplots(figsize=(5.5, 5.2))
        sc = ax.scatter(X_pts, T_pts, c=W_pts, s=np.clip(S_pts, 8.0, 120.0), cmap="viridis", alpha=0.85)
        plt.colorbar(sc, ax=ax, label=r"$|\alpha_j(t,a,\nu)|$")
        ax.scatter([CFG.x0_center], [0.0], s=90, marker="x", color="red", label="IC center")
        ax.set_xlim(CFG.x_min, CFG.x_max)
        ax.set_ylim(CFG.t_min, CFG.t_max)
        ax.set_xlabel("x")
        ax.set_ylabel("t")
        ax.set_title(f"Dynamic basis centers\n{name}")
        ax.legend(loc="upper right")
        save_path = os.path.join(save_dir, f"dynamic_centers_{name}.png")
        plt.tight_layout()
        plt.savefig(save_path, dpi=300)
        plt.close(fig)


def run_full_evaluation(model, history, test_cases, Nx: int, Nt: int, save_dir: str, device):
    plot_training_history(history, os.path.join(save_dir, "training_loss.png"))
    summary = plot_multicase_fields(model, test_cases, Nx=Nx, Nt=Nt, save_dir=save_dir, device=device)
    plot_time_slices(model, test_cases, t_slices=(0.0, 0.125, 0.25, 0.375, 0.5), Nx=400, save_dir=save_dir, device=device)
    plot_error_summary(summary, save_dir)
    save_summary_csv(summary, save_dir)
    visualize_dynamic_centers(model, test_cases, K=60, save_dir=save_dir, device=device)
    return summary


# ============================================================
# Training
# ============================================================
def train_model(args, device):
    model = DynamicBasisMetaSPINN_AdvecDiff(M=args.M, hidden=args.hidden, num_freqs=args.num_freqs).to(device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=max(args.epochs // 3, 1), gamma=0.5)

    best_loss = float("inf")
    best_state = None

    history = {
        "epoch": [],
        "loss": [],
        "loss_pde": [],
        "loss_ic": [],
        "loss_bc": [],
        "loss_reg": [],
    }

    for ep in range(1, args.epochs + 1):
        model.train()
        optimizer.zero_grad()

        # viscosity curriculum
        if ep <= args.epochs // 2:
            nu_range = (CFG.nu_easy, CFG.nu_max)
        else:
            nu_range = (CFG.nu_min, CFG.nu_max)

        # interior: mixture of uniform + localized packet-aware samples
        xt_uniform = sample_interior_points(args.N_int_uniform, nu_range, device)
        xt_local = sample_localized_interior_points(args.N_int_local, nu_range, device)
        xt_int = torch.cat([xt_uniform, xt_local], dim=0)

        xt_ic = sample_ic_points(args.N_ic, nu_range, device)
        xtL, xtR = sample_bc_points(args.N_bc, nu_range, device)

        # PDE
        res = advecdiff_residual(model, xt_int)
        loss_pde = torch.mean(res ** 2)

        # IC (exactly hard-enforced by ansatz, but we still keep a tiny consistency term)
        u_ic = model(xt_ic)
        x_ic = xt_ic[:, 0:1]
        nu_ic = xt_ic[:, 3:4]
        u_ic_true = initial_condition_torch(x_ic, nu_ic).squeeze(-1)
        loss_ic = torch.mean((u_ic - u_ic_true) ** 2)

        # BC from exact benchmark solution
        uL = model(xtL)
        uR = model(xtR)
        uL_true = exact_advecdiff_torch(xtL[:, 0:1], xtL[:, 1:2], xtL[:, 2:3], xtL[:, 3:4]).squeeze(-1)
        uR_true = exact_advecdiff_torch(xtR[:, 0:1], xtR[:, 1:2], xtR[:, 2:3], xtR[:, 3:4]).squeeze(-1)
        loss_bc = torch.mean((uL - uL_true) ** 2) + torch.mean((uR - uR_true) ** 2)

        # light regularization to avoid pathological widths / amplitudes
        aux_t = torch.rand(args.N_reg, 1, device=device) * (CFG.t_max - CFG.t_min) + CFG.t_min
        aux_a, aux_nu = sample_a_nu(args.N_reg, nu_range, device)
        alpha, _, h = model.dynamic_params(aux_t, aux_a, aux_nu)
        loss_reg = 1e-5 * torch.mean(alpha ** 2) + 1e-5 * torch.mean(1.0 / (h ** 2))

        loss = args.lambda_pde * loss_pde + args.lambda_ic * loss_ic + args.lambda_bc * loss_bc + loss_reg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        history["epoch"].append(ep)
        history["loss"].append(loss.item())
        history["loss_pde"].append(loss_pde.item())
        history["loss_ic"].append(loss_ic.item())
        history["loss_bc"].append(loss_bc.item())
        history["loss_reg"].append(loss_reg.item())

        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % args.print_every == 0 or ep == 1:
            print(
                f"Epoch {ep:5d} | Loss {loss.item():.3e} | "
                f"PDE {loss_pde.item():.3e} | IC {loss_ic.item():.3e} | "
                f"BC {loss_bc.item():.3e} | Reg {loss_reg.item():.3e} | "
                f"nu_range=[{nu_range[0]:.3f},{nu_range[1]:.3f}]"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, history



# ============================================================
# Publication-style predictor-corrector plots
# ============================================================
def set_pub_rcparams(fontsize: int = 12, use_serif: bool = False):
    plt.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize + 1,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.titlesize": fontsize + 3,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.9,
        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,
        "mathtext.fontset": "dejavusans",
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.grid": False,
        "lines.linewidth": 2.0,
    })
    if use_serif:
        plt.rcParams["font.family"] = "serif"
        plt.rcParams["mathtext.fontset"] = "dejavuserif"


def case_title(a_test: float, nu_test: float) -> str:
    return rf"$(a,\nu)=({a_test:.2f},{nu_test:.3f})$"


def evaluate_dynamic_coeff_importance_on_grid(hidden_geom, coeffs_dynamic, Nx: int = 220, Nt: int = 220):
    if coeffs_dynamic.size == 0:
        return np.array([])
    x = np.linspace(CFG.x_min, CFG.x_max, Nx)
    t = np.linspace(CFG.t_min, CFG.t_max, Nt)
    X, T = np.meshgrid(x, t, indexing="ij")
    Phi_dyn = eval_spacetime_rbf_np(
        X.ravel(),
        T.ravel(),
        hidden_geom["mu_x"],
        hidden_geom["mu_t"],
        hidden_geom["sigma_x"],
        hidden_geom["sigma_t"],
    )
    contrib = np.mean(np.abs(Phi_dyn * coeffs_dynamic[None, :]), axis=0)
    return contrib


def plot_standardized_predictor_corrector(
    model,
    test_cases,
    device,
    out_dir,
    n_time_guides: int = 16,
    n_dyn_keep: int = 10,
    n_grad_keep: int = 10,
    n_fix_x: int = 18,
    n_fix_t: int = 12,
    N_res: int = 6000,
    N_bc: int = 1200,
    lambda_ridge: float = 1e-8,
    Nx: int = 220,
    Nt: int = 220,
    cmap_solution: str = "viridis",
    cmap_error: str = "magma",
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    predictor_results = []
    corrector_results = []
    summary = []

    for name, a_test, nu_test in test_cases:
        pred_res = evaluate_case_on_grid(model, a_test, nu_test, Nx=Nx, Nt=Nt, device=device)
        hidden_geom = build_refined_frozen_hidden_geometry(
            model=model,
            a_test=a_test,
            nu_test=nu_test,
            device=device,
            n_time_guides=n_time_guides,
            n_dyn_keep=n_dyn_keep,
            n_grad_keep=n_grad_keep,
            n_fix_x=n_fix_x,
            n_fix_t=n_fix_t,
        )
        coeffs, info = solve_corrector_for_case(
            a_test=a_test,
            nu_test=nu_test,
            hidden_geom=hidden_geom,
            N_res=N_res,
            N_bc=N_bc,
            lambda_ridge=lambda_ridge,
            seed=SEED,
        )
        corr_res = evaluate_corrected_case_on_grid(
            a_test=a_test,
            nu_test=nu_test,
            hidden_geom=hidden_geom,
            coeffs=coeffs,
            Nx=Nx,
            Nt=Nt,
        )
        predictor_results.append(pred_res)
        corrector_results.append(corr_res)
        summary.append(
            {
                "name": name,
                "a": a_test,
                "nu": nu_test,
                "predictor_rel_l2": pred_res["rel_l2"],
                "corrector_rel_l2": corr_res["rel_l2"],
                "n_hidden": info["n_hidden"],
            }
        )

    sol_vals = np.concatenate(
        [r["u_exact"].ravel() for r in predictor_results]
        + [r["u_pred"].ravel() for r in predictor_results]
        + [r["u_corr"].ravel() for r in corrector_results]
    )
    sol_vmin = float(np.min(sol_vals))
    sol_vmax = float(np.max(sol_vals))

    err_vals = np.concatenate([r["abs_err"].ravel() for r in corrector_results])
    err_vmax = max(float(np.max(err_vals)), 1e-12)

    ncases = len(test_cases)
    fig, axs = plt.subplots(4, ncases, figsize=(4.2 * ncases, 10.8), constrained_layout=True)
    axs = np.asarray(axs)
    if ncases == 1:
        axs = axs.reshape(4, 1)

    extent = [CFG.x_min, CFG.x_max, CFG.t_min, CFG.t_max]
    im_sol = None
    im_err = None

    for j, ((name, a_test, nu_test), pred_res, corr_res) in enumerate(zip(test_cases, predictor_results, corrector_results)):
        ax = axs[0, j]
        im_sol = ax.imshow(
            pred_res["u_exact"].T,
            origin="lower",
            extent=extent,
            aspect="auto",
            vmin=sol_vmin,
            vmax=sol_vmax,
            cmap=cmap_solution,
        )
        ax.set_title(case_title(a_test, nu_test), pad=6)
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel("Exact")
        else:
            ax.set_yticks([])

        ax = axs[1, j]
        ax.imshow(
            pred_res["u_pred"].T,
            origin="lower",
            extent=extent,
            aspect="auto",
            vmin=sol_vmin,
            vmax=sol_vmax,
            cmap=cmap_solution,
        )
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$u_{\mathrm{Predictor}}$")
        else:
            ax.set_yticks([])

        ax = axs[2, j]
        ax.imshow(
            corr_res["u_corr"].T,
            origin="lower",
            extent=extent,
            aspect="auto",
            vmin=sol_vmin,
            vmax=sol_vmax,
            cmap=cmap_solution,
        )
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$u_{\mathrm{KAPI\!-\!PIELM}}$")
        else:
            ax.set_yticks([])

        ax = axs[3, j]
        im_err = ax.imshow(
            corr_res["abs_err"].T,
            origin="lower",
            extent=extent,
            aspect="auto",
            vmin=0.0,
            vmax=err_vmax,
            cmap=cmap_error,
        )
        ax.set_xlabel("x")
        if j == 0:
            ax.set_ylabel(r"$|u_{\mathrm{KAPI\!-\!PIELM}}-u_{\mathrm{exact}}|$")
        else:
            ax.set_yticks([])

        for i in range(4):
            axs[i, j].set_xticks([CFG.x_min, 0.5 * (CFG.x_min + CFG.x_max), CFG.x_max])
            if j == 0:
                axs[i, j].set_yticks([CFG.t_min, 0.5 * (CFG.t_min + CFG.t_max), CFG.t_max])

    cbar1 = fig.colorbar(im_sol, ax=axs[0:3, :], shrink=0.94, location="right")
    cbar1.set_label(r"$u(x,t)$")

    cbar2 = fig.colorbar(im_err, ax=axs[3, :], shrink=0.94, location="right")
    cbar2.set_label("Absolute Error")

    fig.suptitle("Advection-Diffusion: Exact, Predictor, and Refined KAPI-PIELM Solutions", y=1.02)
    fig.text(
        0.5,
        -0.01,
        rf"Training ranges: $a\in[{CFG.a_min:.2f},{CFG.a_max:.2f}]$, $\nu\in[{CFG.nu_min:.3f},{CFG.nu_max:.3f}]$.",
        ha="center",
        va="top",
    )

    png_path = out_dir / "advecdiff_predictor_corrector_fields.png"
    pdf_path = out_dir / "advecdiff_predictor_corrector_fields.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    csv_path = out_dir / "advecdiff_predictor_corrector_summary.csv"
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["name", "a", "nu", "predictor_rel_l2", "corrector_rel_l2", "n_hidden"],
        )
        writer.writeheader()
        for row in summary:
            writer.writerow(row)

    return png_path, pdf_path, csv_path, summary


def plot_interpretability_predictor_corrector_dynamic_only(
    model,
    test_cases,
    device,
    out_dir,
    n_time_guides: int = 32,#16,
    n_dyn_keep: int = 16,#10,
    n_grad_keep: int = 10,
    n_fix_x: int = 18,
    n_fix_t: int = 12,
    N_res: int = 6000,
    N_bc: int = 1200,
    lambda_ridge: float = 1e-8,
    Nx: int = 220,
    Nt: int = 220,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    cases_data = []
    all_contrib = []

    for _, a_test, nu_test in test_cases:
        hidden_full = build_refined_frozen_hidden_geometry(
            model=model,
            a_test=a_test,
            nu_test=nu_test,
            device=device,
            n_time_guides=n_time_guides,
            n_dyn_keep=n_dyn_keep,
            n_grad_keep=n_grad_keep,
            n_fix_x=n_fix_x,
            n_fix_t=n_fix_t,
        )
        coeffs, _ = solve_corrector_for_case(
            a_test=a_test,
            nu_test=nu_test,
            hidden_geom=hidden_full,
            N_res=N_res,
            N_bc=N_bc,
            lambda_ridge=lambda_ridge,
            seed=SEED,
        )

        dyn_mu_x, dyn_mu_t, dyn_sigma_x, dyn_sigma_t = extract_dynamic_centers_from_predictor(
            model=model,
            a_test=a_test,
            nu_test=nu_test,
            device=device,
            n_time_guides=n_time_guides,
            n_dyn_keep=n_dyn_keep,
            sigma_t_scale=1.25,
        )
        dyn_geom = {
            "mu_x": dyn_mu_x,
            "mu_t": dyn_mu_t,
            "sigma_x": dyn_sigma_x,
            "sigma_t": dyn_sigma_t,
        }

        n_dyn_total = dyn_mu_x.size
        coeffs_dynamic = coeffs[:n_dyn_total]
        contrib_dyn = evaluate_dynamic_coeff_importance_on_grid(
            hidden_geom=dyn_geom,
            coeffs_dynamic=coeffs_dynamic,
            Nx=Nx,
            Nt=Nt,
        )

        cases_data.append((a_test, nu_test, dyn_geom, contrib_dyn))
        all_contrib.append(contrib_dyn)

    all_contrib = np.concatenate(all_contrib) if all_contrib else np.array([0.0, 1.0])
    cmin = float(np.min(all_contrib))
    cmax = float(np.max(all_contrib))
    if cmax <= cmin:
        cmax = cmin + 1e-12

    n_cases = len(cases_data)
    ncols = 2 if n_cases > 1 else 1
    nrows = int(np.ceil(n_cases / ncols))
    fig, axs = plt.subplots(nrows, ncols, figsize=(5.0 * ncols, 3.9 * nrows), constrained_layout=True)
    axs = np.atleast_1d(axs).reshape(nrows, ncols)
    scatter_ref = None

    for k, (a_test, nu_test, dyn_geom, contrib_dyn) in enumerate(cases_data):
        r, c = divmod(k, ncols)
        ax = axs[r, c]
        scatter_ref = ax.scatter(
            dyn_geom["mu_x"],
            dyn_geom["mu_t"],
            c=contrib_dyn,
            s=30,
            cmap="viridis",
            vmin=cmin,
            vmax=cmax,
            alpha=0.88,
            edgecolors="none",
        )
        ax.scatter([CFG.x0_center], [0.0], color="red", marker="x", s=90, linewidths=1.8,
                   label="IC center" if k == 0 else None, zorder=5)
        ax.set_title(case_title(a_test, nu_test), pad=6)
        ax.set_xlim(CFG.x_min, CFG.x_max)
        ax.set_ylim(CFG.t_min, CFG.t_max)
        ax.set_xticks([0.0, 0.5, 1.0])
        ax.set_yticks([CFG.t_min, 0.25, CFG.t_max] if c == 0 else [])
        ax.set_xlabel("x")
        if c == 0:
            ax.set_ylabel("t")
        if k == 0:
            ax.legend(loc="upper right", frameon=True)

    for k in range(n_cases, nrows * ncols):
        r, c = divmod(k, ncols)
        axs[r, c].axis("off")

    cbar = fig.colorbar(scatter_ref, ax=axs, shrink=0.94, pad=0.02)
    cbar.set_label("Mean contribution of dynamic RBF")

    fig.suptitle("Advection-Diffusion: Dynamic Predictor Geometry Used by the Corrector", y=1.02)
    fig.text(
        0.5, -0.01,
        rf"Training ranges: $a\in[{CFG.a_min:.2f},{CFG.a_max:.2f}]$, $\nu\in[{CFG.nu_min:.3f},{CFG.nu_max:.3f}]$.",
        ha="center", va="top"
    )

    png_path = out_dir / "advecdiff_predictor_corrector_interpretability_dynamic.png"
    pdf_path = out_dir / "advecdiff_predictor_corrector_interpretability_dynamic.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    return png_path, pdf_path


# ============================================================
# Main: train once and save model
# ============================================================
def main():
    parser = argparse.ArgumentParser(description="Dynamic-basis meta-SPINN for parametric advection-diffusion (training only)")
    parser.add_argument("--gpu", action="store_true", help="Use CUDA if available")
    parser.add_argument("--epochs", type=int, default=4000)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--M", type=int, default=48)
    parser.add_argument("--hidden", type=int, default=128)
    parser.add_argument("--num_freqs", type=int, default=4)
    parser.add_argument("--N_int_uniform", type=int, default=1024)
    parser.add_argument("--N_int_local", type=int, default=1024)
    parser.add_argument("--N_ic", type=int, default=256)
    parser.add_argument("--N_bc", type=int, default=256)
    parser.add_argument("--N_reg", type=int, default=128)
    parser.add_argument("--lambda_pde", type=float, default=1.0)
    parser.add_argument("--lambda_ic", type=float, default=1.0)
    parser.add_argument("--lambda_bc", type=float, default=1.0)
    parser.add_argument("--Nx", type=int, default=200)
    parser.add_argument("--Nt", type=int, default=200)
    parser.add_argument("--print_every", type=int, default=100)
    parser.add_argument("--outdir", type=str, default="TC_03_FIGURES")
    parser.add_argument("--fontsize", type=int, default=12)
    parser.add_argument("--use_serif", action="store_true")
    parser.add_argument("--model_path", type=str, default="meta_spinn_advecdiff_dynamic.pt")
    args, _unknown = parser.parse_known_args()

    device = torch.device("cuda" if (args.gpu and torch.cuda.is_available()) else "cpu")
    print("Using device:", device)
    set_pub_rcparams(fontsize=args.fontsize, use_serif=args.use_serif)
    os.makedirs(args.outdir, exist_ok=True)

    model, history = train_model(args, device)

    model_path = os.path.join(args.outdir, args.model_path)
    torch.save(model.state_dict(), model_path)
    print(f"Model saved to: {model_path}")

    history_path = os.path.join(args.outdir, "training_history.npz")
    np.savez_compressed(
        history_path,
        epoch=np.asarray(history["epoch"]),
        loss=np.asarray(history["loss"]),
        loss_pde=np.asarray(history["loss_pde"]),
        loss_ic=np.asarray(history["loss_ic"]),
        loss_bc=np.asarray(history["loss_bc"]),
        loss_reg=np.asarray(history["loss_reg"]),
    )
    print(f"Training history saved to: {history_path}")

    plot_training_history(history, os.path.join(args.outdir, "training_loss.png"))


if __name__ == "__main__":
    main()


Using device: cpu
Epoch     1 | Loss 5.143e+00 | PDE 5.082e+00 | IC 0.000e+00 | BC 5.988e-02 | Reg 2.989e-04 | nu_range=[0.030,0.050]
Epoch   100 | Loss 1.815e-01 | PDE 1.769e-01 | IC 0.000e+00 | BC 3.791e-03 | Reg 8.756e-04 | nu_range=[0.030,0.050]
Epoch   200 | Loss 7.232e-02 | PDE 7.000e-02 | IC 0.000e+00 | BC 1.433e-03 | Reg 8.890e-04 | nu_range=[0.030,0.050]
Epoch   300 | Loss 5.643e-02 | PDE 5.472e-02 | IC 0.000e+00 | BC 8.625e-04 | Reg 8.484e-04 | nu_range=[0.030,0.050]
Epoch   400 | Loss 4.037e-02 | PDE 3.883e-02 | IC 0.000e+00 | BC 7.031e-04 | Reg 8.393e-04 | nu_range=[0.030,0.050]
Epoch   500 | Loss 1.898e-02 | PDE 1.770e-02 | IC 0.000e+00 | BC 4.805e-04 | Reg 8.005e-04 | nu_range=[0.030,0.050]
Epoch   600 | Loss 2.153e-02 | PDE 2.044e-02 | IC 0.000e+00 | BC 2.874e-04 | Reg 8.015e-04 | nu_range=[0.030,0.050]
Epoch   700 | Loss 1.573e-02 | PDE 1.475e-02 | IC 0.000e+00 | BC 2.367e-04 | Reg 7.411e-04 | nu_range=[0.030,0.050]
Epoch   800 | Loss 1.409e-02 | PDE 1.320e-02 | IC 0.00